# SARIMA Model Testing
This notebook tests SARIMA models for time series forecasting.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import itertools

# Set random seed for reproducibility
np.random.seed(42)

## Load Data

In [ ]:
# Load dataset
df = pd.read_feather('../dataset/data_andre.feather')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)

## Stationarity Test (ADF Test)

In [ ]:
def adf_test(series, name=''):
    """
    Perform Augmented Dickey-Fuller test to check for stationarity
    """
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'ADF Test Results for {name}:')
    print(f'  ADF Statistic: {result[0]:.6f}')
    print(f'  p-value: {result[1]:.6f}')
    print(f'  Critical Values:')
    for key, value in result[4].items():
        print(f'    {key}: {value:.3f}')
    
    if result[1] <= 0.05:
        print(f'  => Series is STATIONARY (reject null hypothesis)\n')
        return True
    else:
        print(f'  => Series is NON-STATIONARY (fail to reject null hypothesis)\n')
        return False

In [ ]:
# Test stationarity for each column
stationaries = {}
for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        stationaries[col] = adf_test(df[col], name=col)

## ACF and PACF Plots

In [ ]:
# Select target variable for detailed analysis
# Update this to match your dataset
target = df.columns[0]  # Change this to your target column
print(f"Target variable: {target}")

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(df[target].dropna(), lags=40, ax=axes[0])
plot_pacf(df[target].dropna(), lags=40, ax=axes[1])
plt.tight_layout()
plt.show()

## Seasonal Decomposition

In [ ]:
# Seasonal decomposition
try:
    decomposition = seasonal_decompose(df[target], model='additive', period=12)  # Adjust period as needed
    
    fig, axes = plt.subplots(4, 1, figsize=(12, 10))
    decomposition.observed.plot(ax=axes[0], title='Observed')
    decomposition.trend.plot(ax=axes[1], title='Trend')
    decomposition.seasonal.plot(ax=axes[2], title='Seasonal')
    decomposition.resid.plot(ax=axes[3], title='Residual')
    
    for ax in axes:
        ax.set_ylabel('Value')
    axes[-1].set_xlabel('Date')
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not perform seasonal decomposition: {e}")

## Grid Search for SARIMA Parameters

In [ ]:
def grid_search_sarima(data, train_size=500, val_size=100, forecast_window=161, 
                       p_range=range(0, 3), d_range=range(0, 2), q_range=range(0, 3),
                       P_range=range(0, 2), D_range=range(0, 2), Q_range=range(0, 2),
                       s=12):
    """
    Grid search for optimal SARIMA parameters
    (p,d,q)x(P,D,Q,s) parameters
    """
    total_train_val = train_size + val_size
    train = data[-(total_train_val+forecast_window):-(val_size+forecast_window)].values
    val = data[-(val_size+forecast_window):-forecast_window].values
    test = data[-forecast_window:].values
    
    best_mae = float('inf')
    best_params = None
    best_seasonal_params = None
    results = []
    
    total_combinations = (len(p_range) * len(d_range) * len(q_range) * 
                         len(P_range) * len(D_range) * len(Q_range))
    print(f"Testing {total_combinations} SARIMA parameter combinations...\n")
    
    count = 0
    for p, d, q in itertools.product(p_range, d_range, q_range):
        for P, D, Q in itertools.product(P_range, D_range, Q_range):
            try:
                count += 1
                if count % 20 == 0:
                    print(f"Tested {count}/{total_combinations} combinations...")
                
                # Fit ARIMA model (SARIMA with s=seasonal period)
                model = ARIMA(train, order=(p, d, q), 
                             seasonal_order=(P, D, Q, s),
                             disp=False)
                fitted_model = model.fit()
                
                # Predict on validation set
                val_predictions = fitted_model.get_forecast(steps=len(val)).predicted_mean.values
                
                # Calculate MAE
                mae = mean_absolute_error(val, val_predictions)
                rmse = np.sqrt(mean_squared_error(val, val_predictions))
                mape = mean_absolute_percentage_error(val, val_predictions)
                
                results.append({
                    'params': (p, d, q),
                    'seasonal_params': (P, D, Q),
                    'MAE': mae,
                    'RMSE': rmse,
                    'MAPE': mape
                })
                
                if mae < best_mae:
                    best_mae = mae
                    best_params = (p, d, q)
                    best_seasonal_params = (P, D, Q)
                    
            except Exception as e:
                pass  # Skip invalid parameter combinations
    
    return results, best_params, best_seasonal_params

In [ ]:
# Run grid search
results, best_params, best_seasonal_params = grid_search_sarima(
    df[target].values,
    train_size=500,
    val_size=100,
    forecast_window=161,
    p_range=range(0, 3),
    d_range=range(0, 2),
    q_range=range(0, 3),
    P_range=range(0, 2),
    D_range=range(0, 2),
    Q_range=range(0, 2),
    s=12
)

print(f"\nBest SARIMA parameters: {best_params} x {best_seasonal_params}")

In [ ]:
# Convert results to DataFrame and show top 10
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('MAE')

print("Top 10 SARIMA parameter combinations (sorted by MAE):")
print(results_df.head(10).to_string())

# Save results
results_df.to_csv('grid_search_results_sarima.csv', index=False)
print("\nResults saved to grid_search_results_sarima.csv")

## Fit Best Model and Evaluate

In [ ]:
# Data splits
train_size = 500
val_size = 100
forecast_window = 161
total_train_val = train_size + val_size

train = df[target][-(total_train_val+forecast_window):-(val_size+forecast_window)].values
val = df[target][-(val_size+forecast_window):-forecast_window].values
test = df[target][-forecast_window:].values

print(f"Train size: {len(train)}")
print(f"Val size: {len(val)}")
print(f"Test size: {len(test)}")

In [ ]:
# Fit best SARIMA model
p, d, q = best_params
P, D, Q = best_seasonal_params
s = 12

print(f"Fitting SARIMA({p},{d},{q})x({P},{D},{Q},{s})...")

# Train on combined train+val data
full_train = np.concatenate([train, val])
model = ARIMA(full_train, order=(p, d, q), seasonal_order=(P, D, Q, s))
fitted_model = model.fit()

print(fitted_model.summary())

In [ ]:
# Make predictions on test set
test_predictions = fitted_model.get_forecast(steps=len(test)).predicted_mean.values

# Calculate metrics
mse = mean_squared_error(test, test_predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test, test_predictions)
mape = mean_absolute_percentage_error(test, test_predictions)

print(f"\nTest Set Metrics:")
print(f"  MSE:  {mse:.6f}")
print(f"  RMSE: {rmse:.6f}")
print(f"  MAE:  {mae:.6f}")
print(f"  MAPE: {mape:.6f}")

## Visualization

In [ ]:
# Plot actual vs predicted
plt.figure(figsize=(14, 6))
plt.plot(test, label='Actual', linewidth=2)
plt.plot(test_predictions, label='SARIMA Predictions', linewidth=2, linestyle='--')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.title(f'SARIMA({p},{d},{q})x({P},{D},{Q},{s}) Test Set Performance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot residuals
residuals = test - test_predictions

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Residuals over time
axes[0].plot(residuals, label='Residuals')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals Over Time')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Histogram of residuals
axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_xlabel('Residual Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nResiduals Statistics:")
print(f"  Mean: {np.mean(residuals):.6f}")
print(f"  Std Dev: {np.std(residuals):.6f}")